In [2]:
from edgar import set_identity, Company
set_identity("kylian.tarde@efrei.net")

TICKERS = ['AAPL', 'GOOG', 'MSFT', 'TSLA', 'META', 'NVDA', 'NFLX']

ten_ks = {}
for ticker in TICKERS:
    company = Company(ticker)
    filings = company.get_filings(form="10-K")
    for filing in filings:
        if filing.form == "10-K":
            ten_ks[ticker] = filing.text()
            break

In [4]:
import tiktoken
import re

enc = tiktoken.get_encoding("cl100k_base")

def extract_sections(text):
    pattern = re.compile(
        r"(?:^|\n)"                         # start of line
        r"(?:ITEM|Item)\s+"                 # "Item " or "ITEM "
        r"(\d+[A-Z]?)"                      # item number: 1, 1A, 7, 7A...
        r"[\.\:\s\-\—]*"                    # separator (dot, colon, dash...)
        r"([^\n]{0,80})",                   # section title (first 80 chars of line)
        re.MULTILINE,
    )
    
    matches = list(pattern.finditer(text))
    
    if not matches:
        return [{"item": "0", "title": "Full Document", "text": text}]
    
    sections = []
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        
        sections.append({
            "item": match.group(1).upper(),
            "title": match.group(2).strip(),
            "text": text[start:end].strip(),
        })
    
    return sections

text = ten_ks["NVDA"]
sections = extract_sections(text)

for s in sections:
    tokens = len(enc.encode(s["text"]))
    print(f"Item {s['item']:>3} | {tokens:>6} tokens | {s['title']}")

Item   1 |   8970 tokens | Business
Item  1A |  20242 tokens | Risk Factors
Item  1B |     12 tokens | Unresolved Staff Comments
Item  1C |    598 tokens | Cybersecurity
Item   2 |    174 tokens | Properties
Item   3 |     45 tokens | Legal Proceedings
Item   4 |     14 tokens | Mine Safety Disclosures
Item   5 |   1348 tokens | Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Pu
Item   6 |      7 tokens | [Reserved]
Item   7 |   7402 tokens | Management's Discussion and Analysis of Financial Condition and Results of Opera
Item  7A |    825 tokens | Quantitative and Qualitative Disclosures about Market Risk
Item   8 |     40 tokens | Financial Statements and Supplementary Data
Item   9 |     21 tokens | Changes in and Disagreements with Accountants on Accounting and Financial Disclo
Item  9A |    614 tokens | Controls and Procedures
Item  9B |    188 tokens | Other Information
Item  9C |     87 tokens | Disclosure Regarding Foreign Jurisdictions that Preven

In [5]:
def chunk_section(
    section_text: str,
    chunk_size: int = 400,
    overlap: int = 50,
) -> list[str]:
    tokens = enc.encode(section_text)
    
    # If section fits in one chunk, return as-is
    if len(tokens) <= chunk_size:
        return [section_text]
    
    # Split on double newlines first (paragraph boundaries)
    paragraphs = section_text.split("\n\n")
    
    chunks = []
    current_chunk = []
    current_tokens = 0
    
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
            
        para_tokens = len(enc.encode(para))
        
        # If a single paragraph exceeds chunk_size, split it with sliding window
        if para_tokens > chunk_size:
            # Flush current buffer first
            if current_chunk:
                chunks.append("\n\n".join(current_chunk))
                current_chunk = []
                current_tokens = 0
            
            # Sliding window on the big paragraph
            tokens_list = enc.encode(para)
            start = 0
            while start < len(tokens_list):
                end = min(start + chunk_size, len(tokens_list))
                chunks.append(enc.decode(tokens_list[start:end]))
                if end >= len(tokens_list):
                    break
                start += chunk_size - overlap
            continue
        
        # Would adding this paragraph exceed the chunk size?
        if current_tokens + para_tokens > chunk_size:
            # Save current chunk
            chunks.append("\n\n".join(current_chunk))
            
            # Start new chunk — keep last paragraph for overlap/continuity
            if overlap > 0 and current_chunk:
                last = current_chunk[-1]
                last_tokens = len(enc.encode(last))
                current_chunk = [last]
                current_tokens = last_tokens
            else:
                current_chunk = []
                current_tokens = 0
        
        current_chunk.append(para)
        current_tokens += para_tokens
    
    # Don't forget the last chunk
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    
    return chunks


def chunk_10k(text: str, ticker: str, chunk_size: int = 400) -> list[dict]:
    sections = extract_sections(text)
    all_chunks = []
    
    for section in sections:
        chunks = chunk_section(section["text"], chunk_size=chunk_size)
        
        for i, chunk_text in enumerate(chunks):
            all_chunks.append({
                "text": chunk_text,
                "metadata": {
                    "ticker": ticker,
                    "item": section["item"],
                    "section_title": section["title"],
                    "chunk_index": i,
                    "total_chunks_in_section": len(chunks),
                    "token_count": len(enc.encode(chunk_text)),
                },
            })
    
    return all_chunks


all_chunks = []
all_metadata = []

for ticker, text in ten_ks.items():
    chunks = chunk_10k(text, ticker)
    texts = [c["text"] for c in chunks]
    
    for chunk in chunks:
        all_chunks.append(chunk["text"])
        all_metadata.append(chunk["metadata"])

In [6]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# ChromaDB can embed for you, but since you already understand
# how embeddings work, let's give it our own embedding function
# so it uses the same model as us.
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-small-en-v1.5",
    device="cpu",
)

# PersistentClient = saves to disk automatically
# Next time you start Python, your data is still there.
client = chromadb.PersistentClient(path="./chroma_db")

# A "collection" is like a table in a database
collection = client.get_or_create_collection(
    name="sec_10k",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},  # use cosine similarity, same as our np.dot approach
)

print(f"Collection has {collection.count()} documents")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection has 0 documents


In [7]:
# Now insert all our chunks
# ChromaDB needs: unique IDs, documents (text), metadata

import hashlib

ids = []
documents = []
metadatas = []

for i, (chunk_text, metadata) in enumerate(zip(all_chunks, all_metadata)):
    chunk_id = hashlib.md5(
        f"{metadata['ticker']}_{metadata['item']}_{i}_{chunk_text[:100]}".encode()
    ).hexdigest()
    
    ids.append(chunk_id)
    documents.append(chunk_text)
    metadatas.append({
        "ticker": metadata["ticker"],
        "item": metadata["item"],
        "section_title": metadata["section_title"],
        "chunk_index": metadata["chunk_index"],
        "token_count": metadata["token_count"],
    })

# Upsert (insert or update if ID exists)
# Batch because ChromaDB has a per-call limit
batch_size = 500
for i in range(0, len(ids), batch_size):
    collection.upsert(
        ids=ids[i:i+batch_size],
        documents=documents[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size],
    )

print(f"Stored {collection.count()} chunks")

Stored 1952 chunks


In [8]:
def search(query: str, top_k: int = 5, where: dict = None) -> list[dict]:
    """
    Search the vector store.
    
    The 'where' parameter is what makes this better than raw numpy:
    you can filter by metadata BEFORE the vector search runs.
    
    ChromaDB does this efficiently — it doesn't scan all vectors
    then filter. It prunes the search space first.
    """
    kwargs = {
        "query_texts": [query],      # ChromaDB embeds this for you
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }
    if where:
        kwargs["where"] = where
    
    results = collection.query(**kwargs)
    
    # Unpack ChromaDB's nested format
    output = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        output.append({
            "text": doc,
            "ticker": meta["ticker"],
            "section": meta["section_title"],
            "item": meta["item"],
            "score": 1 - dist,  # cosine distance → similarity
        })
    
    return output

In [9]:
# 1. Basic search — no filter
print("=" * 60)
print("QUERY: What are NVIDIA's main risk factors?\n")
for r in search("What are NVIDIA's main risk factors?"):
    print(f"  [{r['ticker']}] Item {r['item']} | {r['score']:.3f} | {r['text'][:80]}...")

# 2. With metadata filter — only NVDA, only Risk Factors
print("\n" + "=" * 60)
print("SAME QUERY but filtered: ticker=NVDA, item=1A\n")
for r in search(
    "What are the main risk factors?",
    where={"$and": [{"ticker": "NVDA"}, {"item": "1A"}]},
):
    print(f"  [{r['ticker']}] Item {r['item']} | {r['score']:.3f} | {r['text'][:80]}...")

# 3. Cross-company comparison
print("\n" + "=" * 60)
print("QUERY: How does each company describe their AI strategy?\n")
for r in search("artificial intelligence strategy and investments", top_k=8):
    print(f"  [{r['ticker']}] Item {r['item']} | {r['score']:.3f} | {r['text'][:80]}...")

QUERY: What are NVIDIA's main risk factors?

  [NVDA] Item 7 | 0.754 | Item 7. Management's Discussion and Analysis of Financial Condition and Results ...
  [NVDA] Item 1 | 0.718 | Gamers choose NVIDIA GPUs to enjoy immersive, increasingly cinematic virtual wor...
  [NVDA] Item 1A | 0.713 | • the ability of developers, end customers and other third parties to build, enh...
  [NVDA] Item 1A | 0.710 | Export controls have and are likely in the future to have a disproportionate imp...
  [NVDA] Item 1A | 0.705 | The use of our GPUs for new, mercurial, or trendy applications, has impacted and...

SAME QUERY but filtered: ticker=NVDA, item=1A

  [NVDA] Item 1A | 0.739 | Item 1A. Risk Factors

The following risk factors should be considered in additi...
  [NVDA] Item 1A | 0.655 | Business disruptions could harm our operations, lead to a decline in revenue and...
  [NVDA] Item 1A | 0.635 | • Our operating results may be adversely impacted by additional tax liabilities,...
  [NVDA] Item 1A | 0.

In [ ]:
import requests
import json


def build_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    """
    Build the full prompt with retrieved context.
    
    Why this structure?
    - System-level instructions set the behavior (cite sources, don't hallucinate)
    - Each source is labeled [Source N] so the LLM can reference them
    - The question comes AFTER the context, so the model reads the sources
      before it sees what to do with them (this matters for small models)
    """
    # Format each chunk as a labeled source
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        header = f"[Source {i}] {chunk['ticker']} — Item {chunk['item']}: {chunk['section']}"
        context_parts.append(f"{header}\n{chunk['text']}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are a financial analyst assistant. Answer the question based ONLY on the provided sources from SEC 10-K filings.

Rules:
- Use ONLY information from the sources below. Do not use prior knowledge.
- Cite your sources using [Source N] when making a claim.
- If the sources don't contain enough information, say so explicitly.
- Be precise with financial figures and time periods.
- Do not generate URLs or links. Source citations use [Source N] format only.
CRITICAL: If a source does not contain specific information, do NOT fill in gaps 
from your own knowledge. Say "the sources do not provide details on X" instead.
Only state facts that you can directly tie to a [Source N].

Sources:
{context}

Question: {query}

Answer with citations:"""
    
    return prompt


def ask(query: str, top_k: int = 5, where: dict = None) -> str:
    """
    End-to-end RAG: retrieve → build prompt → generate.
    """
    # Step 1: Retrieve
    chunks = search(query, top_k=top_k, where=where)
    
    # Step 2: Build prompt
    prompt = build_prompt(query, chunks)
    
    # Step 3: Call Ollama
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:3b",   # adjust to your installed model
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.1,   # low = more factual, less creative
                "num_ctx": 4096,      # context window size
            },
        },
        timeout=120,
    )
    response.raise_for_status()
    answer = response.json()["response"]
    
    # Print sources for transparency
    print("=" * 60)
    print(f"QUERY: {query}\n")
    print("SOURCES USED:")
    for i, c in enumerate(chunks, 1):
        print(f"  [{i}] {c['ticker']} Item {c['item']} | score={c['score']:.3f}")
    print()
    print("ANSWER:")
    print(answer)
    print("=" * 60)
    
    return answer

d:\Programs\anaconda3\envs\10-K-RAG\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [11]:
# Simple factual question
ask("What are NVIDIA's main risk factors?")

# Scoped with metadata filter
ask(
    "What are the main risk factors?",
    where={"$and": [{"ticker": "NVDA"}, {"item": "1A"}]},
)

# Cross-company comparison
ask("Compare the revenue models of Apple and Microsoft")

# Question where the answer might NOT be in the data
ask("What is Tesla's dividend policy?")

QUERY: What are NVIDIA's main risk factors?

SOURCES USED:
  [1] NVDA Item 7 | score=0.754
  [2] NVDA Item 1 | score=0.718
  [3] NVDA Item 1A | score=0.713
  [4] NVDA Item 1A | score=0.710
  [5] NVDA Item 1A | score=0.705

ANSWER:
NVIDIA faces several significant risks as outlined in its SEC filings. The primary risks include:

1. **The ability of developers, end customers, and other third parties to build, enhance, and maintain accelerated computing applications that leverage NVIDIA’s platforms** (Source 3).
2. **Changes impacting the ecosystem for architectures underlying their products and technologies** (Source 3).
3. **Government actions or changes in governmental policies such as export controls, increased restrictions on gaming usage, or tariffs** (Source 1 & Source 4).
4. **Customers' and partners’ ability to secure capital and energy and build complex datacenter infrastructure timely** (Source 3).
5. **Availability of third-party content on their platforms, such as GeForce NOW

'Tesla has never declared or paid cash dividends on its common stock. The company currently does not anticipate paying any such cash dividends in the foreseeable future. This information can be found directly in [Source 1] under "Dividend Policy."'

In [12]:
for ticker in TICKERS:
    results = collection.get(
        where={"ticker": ticker},
        limit=1,
        include=["metadatas"],
    )
    count = collection.count()  # total
    # Count per ticker by getting all
    ticker_results = collection.get(
        where={"ticker": ticker},
        include=["metadatas"],
    )
    print(f"  {ticker}: {len(ticker_results['ids'])} chunks")

  AAPL: 168 chunks
  GOOG: 274 chunks
  MSFT: 327 chunks
  TSLA: 302 chunks
  META: 386 chunks
  NVDA: 261 chunks
  NFLX: 234 chunks


In [14]:
ask(
    "Compare the revenue models of Apple and Microsoft",
    where={"ticker": {"$in": ["AAPL", "MSFT"]}},
    top_k=8,  # more chunks to get both companies
)

QUERY: Compare the revenue models of Apple and Microsoft

SOURCES USED:
  [1] MSFT Item 8 | score=0.773
  [2] MSFT Item 7 | score=0.756
  [3] MSFT Item 7 | score=0.751
  [4] MSFT Item 7 | score=0.743
  [5] MSFT Item 8 | score=0.725
  [6] MSFT Item 1A | score=0.722
  [7] MSFT Item 7 | score=0.722
  [8] AAPL Item 8 | score=0.718

ANSWER:
To compare the revenue models of Apple Inc. (AAPL) and Microsoft Corporation (MSFT), we need to analyze their respective financial statements, focusing on key segments such as Products and Services, Revenue Recognition, and other relevant sections.

### Revenue Models Overview:
- **Apple Inc. (AAPL):**
  - AAPL primarily generates revenue from the sale of consumer electronics like iPhones, iPads, Mac computers, and Apple Watches.
  - They also generate significant revenue through their services business, including App Store subscriptions, iCloud storage, and Apple Pay transactions.

- **Microsoft Corporation (MSFT):**
  - MSFT's primary revenue streams c

"To compare the revenue models of Apple Inc. (AAPL) and Microsoft Corporation (MSFT), we need to analyze their respective financial statements, focusing on key segments such as Products and Services, Revenue Recognition, and other relevant sections.\n\n### Revenue Models Overview:\n- **Apple Inc. (AAPL):**\n  - AAPL primarily generates revenue from the sale of consumer electronics like iPhones, iPads, Mac computers, and Apple Watches.\n  - They also generate significant revenue through their services business, including App Store subscriptions, iCloud storage, and Apple Pay transactions.\n\n- **Microsoft Corporation (MSFT):**\n  - MSFT's primary revenue streams come from its cloud computing platform (Microsoft Cloud), productivity software (Office 365), gaming platforms (Xbox Game Pass), and enterprise solutions.\n  - They also generate substantial revenue through their Windows operating system, which is sold to OEMs for integration into devices.\n\n### Revenue Recognition:\n- **AAPL:*

In [15]:
def search_compare(query: str, tickers: list[str], per_ticker: int = 4) -> list[dict]:
    """
    For comparison queries: retrieve N chunks per company separately,
    then merge. This guarantees balanced representation.
    
    Without this, the vector search is biased toward whichever
    company uses vocabulary closest to the query.
    """
    all_results = []
    
    for ticker in tickers:
        results = search(
            query,
            top_k=per_ticker,
            where={"ticker": ticker},
        )
        all_results.extend(results)
    
    # Sort by score so the best chunks from each company interleave
    all_results.sort(key=lambda x: x["score"], reverse=True)
    return all_results

In [16]:
# Test it
results = search_compare(
    "Compare the revenue models",
    tickers=["AAPL", "MSFT"],
    per_ticker=4,
)

for r in results:
    print(f"  [{r['ticker']}] Item {r['item']} | {r['score']:.3f}")

  [MSFT] Item 8 | 0.751
  [MSFT] Item 8 | 0.720
  [MSFT] Item 8 | 0.701
  [MSFT] Item 8 | 0.701
  [AAPL] Item 8 | 0.690
  [AAPL] Item 8 | 0.660
  [AAPL] Item 8 | 0.657
  [AAPL] Item 8 | 0.655
